# 03 — SQL Analytics, Provider Intelligence Scorecard & Unit Economics

## Objective
1. Execute the 11 production SQL queries in `sql/medtoureasy_analysis.sql` against `mte_patient_intelligence.db`.
2. Compute the composite **Provider Intelligence Score** across all 30 partner hospitals.
3. Benchmark 15 care coordinators, 6 acquisition channels, and 9 treatment specialties.

In [1]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

from src.data_quality import load_csvs
from src.analytics import (
    compute_provider_scorecard,
    compute_coordinator_sla,
    compute_channel_performance,
    compute_treatment_economics,
)

In [2]:
conn = sqlite3.connect(DATA / 'mte_patient_intelligence.db')
sql_text = (ROOT / 'sql' / 'medtoureasy_analysis.sql').read_text()
queries = [q.strip() for q in sql_text.split(';') if q.strip() and not all(l.strip().startswith('--') for l in q.strip().splitlines())]
print(f'Loaded {len(queries)} SQL queries from medtoureasy_analysis.sql')
for idx, q in enumerate(queries, 1):
    res = pd.read_sql_query(q, conn)
    print(f'Query {idx:02d} returned {res.shape[0]} rows x {res.shape[1]} cols')
conn.close()

Loaded 11 SQL queries from medtoureasy_analysis.sql
Query 01 returned 1 rows x 10 cols
Query 02 returned 30 rows x 15 cols
Query 03 returned 9 rows x 10 cols
Query 04 returned 15 rows x 11 cols
Query 05 returned 6 rows x 11 cols
Query 06 returned 12 rows x 7 cols
Query 07 returned 35 rows x 5 cols
Query 08 returned 12 rows x 9 cols
Query 09 returned 30 rows x 6 cols
Query 10 returned 3 rows x 10 cols
Query 11 returned 4 rows x 2 cols


## 1. Provider Performance Scorecard & Composite Intelligence Ranking

In [3]:
tables = load_csvs(DATA)
provider_sc = compute_provider_scorecard(
    tables['patient_journey'], tables['provider_master'], tables['country_reference'], tables['treatment_outcomes']
)
print('=== Top 10 Providers by Composite Intelligence Score ===')
display(provider_sc.head(10))
print('=== Bottom 5 Providers (Intervention Candidates) ===')
display(provider_sc.tail(5))

=== Top 10 Providers by Composite Intelligence Score ===
=== Bottom 5 Providers (Intervention Candidates) ===


,provider_id,provider_name,total_assigned_leads,completed_cases,conversion_rate,avg_response_time_hours,avg_engagement_score,sla_breach_rate,follow_up_completion_rate,avg_satisfaction_score,total_gross_margin_inr,sla_adherence_rate,margin_per_lead,conv_score,sla_score,followup_score,satisfaction_score_norm,margin_score,provider_intelligence_score,intelligence_rank
0,PVD-020,MTE Partner Hospital 20,55,29,0.527273,5.854545,61.843636,0.272727,0.381818,7.962069,5844019,0.727273,106254.890909,100.000000,72.727273,38.181818,79.620690,92.796222,80.775075,1
1,PVD-008,MTE Partner Hospital 08,55,28,0.509091,10.280000,61.052727,0.545455,0.145455,8.225000,6145666,0.454545,111739.381818,93.867925,45.454545,14.545455,82.250000,100.000000,71.770605,2
2,PVD-007,MTE Partner Hospital 07,50,26,0.520000,15.406000,55.846000,0.820000,0.240000,7.492308,5452853,0.180000,109057.060000,97.547170,18.000000,24.000000,74.923077,96.476820,66.997976,3
3,PVD-015,MTE Partner Hospital 15,56,26,0.464286,12.510714,57.621429,0.696429,0.250000,8.223077,5975718,0.303571,106709.250000,78.756739,30.357143,25.000000,82.230769,93.393014,64.461668,4
4,PVD-001,MTE Partner Hospital 01,55,26,0.472727,16.783636,49.949091,0.890909,0.363636,7.453846,5496283,0.109091,99932.418182,81.603774,10.909091,36.363636,74.538462,84.491770,60.196619,5
5,PVD-011,MTE Partner Hospital 11,49,23,0.469388,13.563265,59.004082,0.714286,0.224490,7.804348,3810030,0.285714,77755.714286,80.477474,28.571429,22.448980,78.043478,55.363076,56.004012,6
6,PVD-017,MTE Partner Hospital 17,43,20,0.465116,15.230233,58.553488,0.837209,0.093023,7.780000,3703765,0.162791,86134.069767,79.036858,16.279070,9.302326,77.800000,66.367892,53.305799,7
7,PVD-003,MTE Partner Hospital 03,59,26,0.440678,14.893220,58.506780,0.762712,0.288136,7.296154,4713255,0.237288,79885.677966,70.794691,23.728814,28.813559,72.961538,58.160744,52.882584,8
8,PVD-022,MTE Partner Hospital 22,47,20,0.425532,13.344681,59.800000,0.765957,0.170213,7.335000,4154461,0.234043,88392.787234,65.686471,23.404255,17.021277,73.350000,69.334676,51.809419,9
9,PVD-027,MTE Partner Hospital 27,53,21,0.396226,14.971698,54.794340,0.698113,0.245283,7.642857,4352788,0.301887,82128.075472,55.802777,30.188679,24.528302,76.428571,61.106092,50.143318,10


,provider_id,provider_name,total_assigned_leads,completed_cases,conversion_rate,avg_response_time_hours,avg_engagement_score,sla_breach_rate,follow_up_completion_rate,avg_satisfaction_score,total_gross_margin_inr,sla_adherence_rate,margin_per_lead,conv_score,sla_score,followup_score,satisfaction_score_norm,margin_score,provider_intelligence_score,intelligence_rank
25,PVD-025,MTE Partner Hospital 25,52,12,0.230769,18.934615,50.582692,0.903846,0.096154,7.391667,2893200,0.096154,55638.461538,0.000000,9.615385,9.615385,73.916667,26.312471,19.715379,26
26,PVD-028,MTE Partner Hospital 28,39,9,0.230769,17.443590,47.830769,0.897436,0.153846,7.577778,1802458,0.102564,46216.871795,0.000000,10.256410,15.384615,75.777778,13.937385,18.513118,27
27,PVD-014,MTE Partner Hospital 14,52,12,0.230769,16.965385,47.588462,0.903846,0.096154,7.350000,2346610,0.096154,45127.115385,0.000000,9.615385,9.615385,73.500000,12.506010,16.891587,28
28,PVD-013,MTE Partner Hospital 13,50,12,0.240000,25.912000,46.706000,1.000000,0.120000,7.033333,2301864,0.000000,46037.280000,3.113208,0.000000,12.000000,70.333333,13.701495,16.024261,29
29,PVD-023,MTE Partner Hospital 23,46,11,0.239130,25.208696,48.904348,1.000000,0.086957,7.272727,1637869,0.000000,35605.847826,2.819934,0.000000,8.695652,72.727273,0.000000,13.059419,30


## 2. Coordinator SLA & Acquisition Channel Efficiency

In [4]:
coord_df = compute_coordinator_sla(tables['patient_journey'], tables['country_reference'])
display(coord_df)

channel_df = compute_channel_performance(tables['patient_journey'])
display(channel_df)

econ_df = compute_treatment_economics(tables['patient_journey'])
display(econ_df)

,coordinator_id,leads,consultations,completed,followups,avg_response_hours,sla_breach_rate,consultation_rate,completion_rate,followup_rate,sla_adherence_rate
0,COORD-01,100,72,37,19,16.528000,0.840000,0.720000,0.370000,0.190000,0.160000
1,COORD-02,106,65,36,23,17.014151,0.811321,0.613208,0.339623,0.216981,0.188679
2,COORD-03,96,58,40,21,17.363542,0.822917,0.604167,0.416667,0.218750,0.177083
3,COORD-04,112,79,39,22,16.799107,0.794643,0.705357,0.348214,0.196429,0.205357
4,COORD-05,101,65,39,26,16.194059,0.841584,0.643564,0.386139,0.257426,0.158416
5,COORD-06,111,80,47,28,16.281982,0.855856,0.720721,0.423423,0.252252,0.144144
6,COORD-07,106,67,39,20,17.158491,0.830189,0.632075,0.367925,0.188679,0.169811
7,COORD-08,103,60,40,27,17.503883,0.844660,0.582524,0.388350,0.262136,0.155340
8,COORD-09,88,51,27,14,18.335227,0.863636,0.579545,0.306818,0.159091,0.136364
9,COORD-10,100,67,31,19,16.911000,0.880000,0.670000,0.310000,0.190000,0.120000


,lead_source,leads,consultations,quotes,completed,revenue,cost,gross_margin,consultation_rate,completion_rate,margin_per_lead
0,Chatbot,254,178,106,93,43780511,24532697,19247814,0.700787,0.366142,75778.795276
1,Email,238,147,79,95,43172255,23697709,19474546,0.617647,0.399160,81825.823529
2,Phone,244,165,94,94,43555749,23445330,20110419,0.676230,0.385246,82419.750000
3,Referral Partner,246,154,89,82,36184874,20267172,15917702,0.626016,0.333333,64706.105691
4,Web Form,246,152,84,85,35708940,19965359,15743581,0.617886,0.345528,63998.296748
5,WhatsApp,272,188,109,105,45922916,24902622,21020294,0.691176,0.386029,77280.492647


,treatment_category,leads,completed,estimated_case_value_inr,actual_revenue_inr,service_cost_inr,gross_margin_inr,completion_rate,margin_per_completed_case
0,Bariatric Surgery,162,67,71375922,27423142,14855857,12567285,0.413580,187571.417910
1,Cardiology,158,57,74394833,25108555,13771132,11337423,0.360759,198902.157895
2,Cosmetic Surgery,180,63,76201924,16126088,9018062,7108026,0.350000,112825.809524
3,Dental,156,59,51864890,6866889,3817728,3049161,0.378205,51680.694915
4,Fertility,166,54,69761684,16800212,9171244,7628968,0.325301,141277.185185
5,Neurology,174,70,84451051,36111916,19854602,16257314,0.402299,232247.342857
6,Oncology,187,73,98762447,48123276,26348999,21774277,0.390374,298277.767123
7,Orthopedics,156,57,67869338,21397906,11848685,9549221,0.365385,167530.192982
8,Transplant Evaluation,161,54,99662329,50367261,28124580,22242681,0.335404,411901.500000
